# 01 · Connectome tour

What the male CNS looks like as a graph, what the reservoir subgraph looks like, and how its eigenvalue spectrum compares to the control wirings.

Uses the real data if you've run `scripts/download_data.py`; otherwise it falls back to the synthetic stand-in and says so loudly.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # configs and data paths are relative to the repo root

import numpy as np
import pandas as pd

from flyres import plotting
from flyres.connectome import FILES, cache_stem, load_connectome
from flyres.controls import make_wiring
from flyres.reservoir import scale_weights, signed_weights
from flyres.subgraph import graph_stats, select_subgraph
from flyres.synthetic import synthetic_connectome

RAW, CACHE, MIN_WEIGHT = Path("data/raw"), Path("data/cache"), 5
have_real = (CACHE / f"{cache_stem(MIN_WEIGHT)}.npz").exists() or all((RAW / f).exists() for f in FILES.values())
conn = load_connectome(RAW, CACHE, MIN_WEIGHT) if have_real else synthetic_connectome(n=3000, seed=0)
print("REAL male CNS v1.0" if have_real else "SYNTHETIC stand-in: download the data for the real thing")
conn.describe()

## Who's in there

In [ ]:
conn.neurons["superclass"].value_counts().head(20)

In [ ]:
# transmitter mix per superclass (the sign of each neuron's synapses comes from this)
mix = pd.crosstab(conn.neurons["superclass"], conn.neurons["nt"])
mix.loc[mix.sum(axis=1).sort_values(ascending=False).index].head(15)

## Degree distribution

Heavy tails: most neurons have a modest number of partners, a few hubs have thousands. Random (Erdős–Rényi) graphs don't look like this at all, which is one reason the controls matter.

In [ ]:
plotting.plot_degree_ccdf(conn.neurons);

## The reservoir subgraph

Grown from the 100 most-connected sensory neurons by repeatedly adding the neurons most strongly connected to the current set.

In [ ]:
sub = select_subgraph(conn, n_neurons=3000, n_inputs=100, input_filter={"superclass": "sensory"})
cols = [c for c in ["superclass", "class", "type"] if c in sub.neurons.columns]
sub.neurons.loc[sub.input_idx, cols].value_counts().head(10)

In [ ]:
WIRINGS = ["connectome", "degree_preserving", "weight_shuffle", "sign_shuffle", "erdos_renyi"]
rows = []
for w in WIRINGS:
    Wc, s = make_wiring(w, sub.W, sub.sign, np.random.default_rng(0))
    rows.append({"wiring": w, **graph_stats(Wc, s)})
pd.DataFrame(rows).set_index("wiring").round(3)

## Eigenvalue spectra at equal spectral radius

Every wiring is rescaled so its largest |eigenvalue| is 0.9 (the gray circle). What differs is where the rest of the spectrum sits: eigenvalues near the circle are slow modes (long memory), eigenvalues near zero die out fast. Dense eigendecomposition, so give it a minute at 3,000 neurons.

In [ ]:
eigs = {}
for w in ["connectome", "degree_preserving", "erdos_renyi"]:
    Wc, s = make_wiring(w, sub.W, sub.sign, np.random.default_rng(0))
    W, _ = scale_weights(signed_weights(Wc, s, "log1p"), 0.9)
    eigs[w] = np.linalg.eigvals(W.toarray().astype(np.float64))
plotting.plot_spectra(eigs, radius=0.9);

In [ ]:
pd.DataFrame({w: {"share |λ| > 0.5": (np.abs(e) > 0.5).mean(),
                  "share |λ| > 0.2": (np.abs(e) > 0.2).mean(),
                  "median |λ|": np.median(np.abs(e))} for w, e in eigs.items()}).T.round(3)